# Notebook 23 — Information Asymmetry Gradient

2x2 RDD: purchase/refinance x high/low LTV.

**Input:** panel files  **Output:** table_23, figure_23  **Runtime:** ~25 min

In [ ]:

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
PROC = Path("../data/processed"); TABS = Path("../outputs/tables"); FIGS = Path("../outputs/figures")
TABS.mkdir(exist_ok=True); FIGS.mkdir(exist_ok=True)
YEARS=[2020,2021,2022,2023,2024]; BLACK_CODE=3; CUTOFF=80.0; BW=10.0
print("NB23: INFORMATION ASYMMETRY GRADIENT")
print("Prediction: theta largest for high-LTV purchase, near zero for low-LTV refinance")
dfs=[]
for yr in YEARS:
    fp = PROC/f"panel_{yr}.csv"
    if not fp.exists(): continue
    df = pd.read_csv(fp, usecols=["lei","applicant_race_1","approved","income","loan_amount","property_value","ltv","loan_purpose"])
    df["black"]    = (df["applicant_race_1"]==BLACK_CODE).astype(int)
    df["approved"] = pd.to_numeric(df["approved"], errors="coerce")
    df["income"]   = pd.to_numeric(df["income"],   errors="coerce")
    df["ltv"]      = pd.to_numeric(df["ltv"],      errors="coerce")
    df["lp"]       = pd.to_numeric(df["loan_purpose"], errors="coerce")
    if df["ltv"].isna().mean()>0.5:
        df["ltv"] = pd.to_numeric(df["loan_amount"],errors="coerce") / pd.to_numeric(df["property_value"],errors="coerce") * 100
    df = df[(df["ltv"]>=CUTOFF-BW)&(df["ltv"]<CUTOFF+BW)&df["applicant_race_1"].isin([BLACK_CODE,5])].copy()
    df["ltv_c"]=df["ltv"]-CUTOFF; df["above80"]=(df["ltv"]>=CUTOFF).astype(int)
    df["is_purchase"]=(df["lp"]==1).astype(int); df["year"]=yr
    dfs.append(df.dropna(subset=["approved","ltv","income"]))
    print(f"  {yr}: {len(dfs[-1]):,} obs")
df_all = pd.concat(dfs, ignore_index=True)
print(f"Total in bandwidth: {len(df_all):,}")
print(f"  Purchase: {df_all['is_purchase'].sum():,}  Other: {(1-df_all['is_purchase']).sum():,}")


In [ ]:

def run_rdd(df_input, label=""):
    df = df_input.copy()
    lr = df.groupby("lei")["black"].agg(["sum","count"])
    valid = lr[(lr["sum"]>=5)&(lr["count"]-lr["sum"]>=5)].index
    df = df[df["lei"].isin(valid)].copy()
    if len(df)<200: return None
    df["b80"]   = df["black"]*df["above80"]
    df["lc80"]  = df["ltv_c"]*df["above80"]
    df["blc"]   = df["black"]*df["ltv_c"]
    df["blca"]  = df["black"]*df["ltv_c"]*df["above80"]
    regs = ["black","ltv_c","above80","b80","lc80","blc","blca","income","loan_amount"]
    df = df.dropna(subset=["approved"]+regs)
    lm = df.groupby("lei")[["approved"]+regs].transform("mean")
    for c in ["approved"]+regs:
        df[c+"_dm"] = df[c]-lm[c]
    Xc = [c+"_dm" for c in regs]; dr = df[["approved_dm"]+Xc+["lei"]].dropna()
    X=dr[Xc].values; y=dr["approved_dm"].values; lei=dr["lei"].values
    Xf=np.column_stack([np.ones(len(X)),X]); coef,_,_,_=np.linalg.lstsq(Xf,y,rcond=None)
    e=y-Xf@coef; ul=np.unique(lei); G=len(ul); n=len(y); k=Xf.shape[1]
    if G<2: return None
    adj=(G/(G-1))*((n-1)/(n-k)); br=np.linalg.inv(Xf.T@Xf); mt=np.zeros((k,k))
    for lend in ul:
        idx=(lei==lend); sc=Xf[idx].T@e[idx]; mt+=np.outer(sc,sc)
    vcov=adj*br@mt@br; se=np.sqrt(np.diag(vcov))
    cn=["const"]+Xc; ti=cn.index("b80_dm")
    th=coef[ti]*100; tse=se[ti]*100; ts=th/tse if tse>0 else 0
    p=2*(1-stats.t.cdf(abs(ts),df=G-1))
    sig="***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "n.s."
    return {"Label":label,"N_obs":n,"N_lenders":G,"Theta_pp":round(th,3),
            "Theta_SE":round(tse,3),"T_stat":round(ts,3),"P_value":round(p,4),"Sig":sig}

cells = [
    ("Purchase", "Above 80%", (df_all["is_purchase"]==1)&(df_all["ltv"]>=CUTOFF)),
    ("Purchase", "Below 80%", (df_all["is_purchase"]==1)&(df_all["ltv"]< CUTOFF)),
    ("Refinance","Above 80%", (df_all["is_purchase"]==0)&(df_all["ltv"]>=CUTOFF)),
    ("Refinance","Below 80%", (df_all["is_purchase"]==0)&(df_all["ltv"]< CUTOFF)),
]
grad_res = []
print("2x2 GRADIENT RESULTS:")
for lt, lg, mask in cells:
    sub = df_all[mask]
    r = run_rdd(sub, f"{lt} x {lg}")
    if r:
        r["Loan_Type"]=lt; r["LTV_Group"]=lg; grad_res.append(r)
        print(f"  {lt} x {lg}: theta={r['Theta_pp']:+.3f}pp  SE={r['Theta_SE']:.3f}  {r['Sig']}")

df_grad = pd.DataFrame(grad_res)
df_grad.to_csv(TABS/"table_23_asymmetry_gradient.csv", index=False)
print("Saved table_23_asymmetry_gradient.csv")
print()
print("INTERPRETATION: Monotonic gradient (Purch/High > Purch/Low > Refi/High > Refi/Low)?")
if grad_res:
    vals = [r["Theta_pp"] for r in grad_res]
    mono = all(vals[i] <= vals[i+1] for i in range(len(vals)-1)) or all(vals[i] >= vals[i+1] for i in range(len(vals)-1))
    print(f"  Values: {[round(v,2) for v in vals]}")
    print(f"  Monotonic: {mono}")


In [ ]:

fig, axes = plt.subplots(1,2,figsize=(14,6))
ax = axes[0]
labels = [r["Label"] for r in grad_res]
thetas = [r["Theta_pp"] for r in grad_res]
ses    = [r["Theta_SE"] for r in grad_res]
sigs   = [r["Sig"] for r in grad_res]
colors = ["#B71C1C","#E57373","#1565C0","#90CAF9"]
ax.bar(range(len(labels)),thetas,color=colors,alpha=0.85,edgecolor="white")
ax.errorbar(range(len(labels)),thetas,yerr=[1.96*s for s in ses],fmt="none",color="black",capsize=6,capthick=2)
for i,(t,s) in enumerate(zip(thetas,sigs)):
    ax.text(i,t-0.15,f"{t:.2f}pp {s}",ha="center",fontsize=8.5,fontweight="bold")
ax.axhline(0,color="black",linewidth=1.2)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(["Purch/High","Purch/Low","Refi/High","Refi/Low"],fontsize=9)
ax.set_ylabel("theta: RDD Threshold Effect (pp)",fontsize=10)
ax.set_title("Panel A: 2x2 Information Asymmetry Gradient\nLargest where PMI and asymmetry greatest",fontsize=11,fontweight="bold")
ax.grid(alpha=0.3,axis="y")

ax = axes[1]
if len(grad_res)==4:
    matrix = np.array([[grad_res[0]["Theta_pp"],grad_res[1]["Theta_pp"]],
                       [grad_res[2]["Theta_pp"],grad_res[3]["Theta_pp"]]])
    vabs = max(abs(t) for t in thetas)+0.5
    im = ax.imshow(matrix,cmap="RdYlBu_r",aspect="auto",vmin=-vabs,vmax=0.5)
    plt.colorbar(im,ax=ax,label="theta (pp)")
    for i in range(2):
        for j in range(2):
            r = grad_res[i*2+j]
            col = "white" if abs(r["Theta_pp"])>1 else "black"
            ax.text(j,i,f"{r['Theta_pp']:.2f}pp\n{r['Sig']}",
                    ha="center",va="center",fontsize=11,fontweight="bold",color=col)
    ax.set_xticks([0,1]); ax.set_xticklabels(["Above 80%","Below 80%"],fontsize=10)
    ax.set_yticks([0,1]); ax.set_yticklabels(["Purchase","Refinance"],fontsize=10)
    ax.set_title("Panel B: Gradient Heatmap\n(darker red = larger negative theta)",fontsize=11,fontweight="bold")

fig.suptitle("Figure 23: Information Asymmetry Gradient\nRDD effect by loan purpose and LTV group",fontsize=12,fontweight="bold")
plt.tight_layout()
plt.savefig(FIGS/"figure_23_gradient.png",dpi=300,bbox_inches="tight")
plt.show()
print("Saved figure_23_gradient.png  |  NB23 COMPLETE")
